# Clean Data
Takes the merged file (merged_clean.csv), fixes the last data problems, and adds new columns. Saves games_final.csv, which is the file every analysis notebook should load.

## Setup

In [1]:
import pandas as pd

CLEAN = "../data/clean/"

In [2]:
df = pd.read_csv(CLEAN + "merged_clean.csv", low_memory=False)
print(df.shape)

(30807, 32)


## Remove impossible platforms

Metacritic's file has some games listed on consoles that were long dead when the game came out, like *Predator: Hunting Grounds* on PSP (2020), *Cloudpunk* on Wii (2020), and *Harvest Moon: One World* on GameCube (2021).

For each older console, set the last year it realistically got new games. Drop **Metacritic only** rows released after that year. VG rows are left alone, since those have real sales records.

In [3]:
last_year = {
    'NES': 1995, 'SNES': 2000, 'GB': 2003, 'N64': 2002, 'GBA': 2008, 'GC': 2007,
    'DS': 2014, 'Wii': 2019, 'WiiU': 2019, '3DS': 2020,
    'PS': 2006, 'PS2': 2013, 'PS3': 2019, 'PSP': 2014, 'PSV': 2020,
    'XB': 2008, 'X360': 2018, 'DC': 2002,
}
impossible = (df['year'] > df['platform'].map(last_year)) & (df['mc_match'] == 'mc only')

display(df.loc[impossible, ['name', 'platform', 'year', 'critic_score']].sample(10, random_state=0))
print(f"Impossible Metacritic rows: {impossible.sum()}")

,name,platform,year,critic_score
6322,Fear Effect Sedna,XB,2018.0,42.0
806,Arcania: Fall of Setarrif,N64,2011.0,41.0
9423,Jump Force: Deluxe Edition,X360,2020.0,50.0
18734,Strobophagia | Rave Horror,Wii,2020.0,52.0
4102,Dead Rising 3: Operation Broken Eagle,PS2,2014.0,52.0
18623,Stormrise,GC,2009.0,48.0
16495,Rush Bros,GC,2013.0,54.0
16483,RUNE II: Decapitation Edition,DS,2019.0,50.0
22222,Waking,PS2,2020.0,44.0
4948,Doodle God: Ultimate Edition,PSP,2017.0,53.0


Impossible Metacritic rows: 281


In [4]:
before = len(df)
df = df[~impossible]
print(f"Rows: {before} -> {len(df)}")

Rows: 30807 -> 30526


## Fix a year error in VG

In [7]:
fix = (df['name'] == 'Imagine: Makeup Artist') & (df['platform'] == 'DS')
df.loc[fix, 'year'] = 2009
df.loc[fix, ['name', 'platform', 'year']]

,name,platform,year
8797,Imagine: Makeup Artist,DS,2009.0


## Small inconsistencies

- **Names:** a few have extra spaces at the start or end, so we strip them.
- **ESRB rating:** `K-A` ("Kids to Adults") is the old name for `E`, so they're the same rating. `RP` means "Rating Pending," which isn't a real rating, so it becomes missing.

In [5]:
df['name'] = df['name'].str.strip()

df['esrb_rating'] = df['esrb_rating'].replace({'K-A': 'E', 'RP': pd.NA})
df['esrb_rating'].value_counts(dropna=False)

esrb_rating
NaN     20584
E        3993
T        2958
M        1563
E10+     1419
EC          8
AO          1
Name: count, dtype: int64

## Group platforms into companies

In [8]:
platform_group = {
    'PC': 'PC',
    # Sony
    'PS': 'PlayStation', 'PS2': 'PlayStation', 'PS3': 'PlayStation', 'PS4': 'PlayStation',
    'PS5': 'PlayStation', 'PSP': 'PlayStation', 'PSV': 'PlayStation',
    # Microsoft
    'XB': 'Xbox', 'X360': 'Xbox', 'XOne': 'Xbox', 'XboxSX': 'Xbox',
    # Nintendo
    'NES': 'Nintendo', 'SNES': 'Nintendo', 'N64': 'Nintendo', 'GC': 'Nintendo', 'Wii': 'Nintendo',
    'WiiU': 'Nintendo', 'Switch': 'Nintendo', 'GB': 'Nintendo', 'GBA': 'Nintendo',
    'DS': 'Nintendo', '3DS': 'Nintendo',
    # everything else
    'GEN': 'Other', 'SAT': 'Other', 'DC': 'Other', 'GG': 'Other', 'SCD': 'Other',
    '2600': 'Other', 'NG': 'Other', 'TG16': 'Other', '3DO': 'Other', 'WS': 'Other',
    'PCFX': 'Other', 'Stadia': 'Other',
}
df['platform_group'] = df['platform'].map(platform_group)

print("Unmapped platforms:", df['platform_group'].isna().sum())   # should be 0
df['platform_group'].value_counts()

Unmapped platforms: 0


platform_group
PlayStation    9741
Nintendo       8599
PC             7281
Xbox           4382
Other           523
Name: count, dtype: int64

## Put scores on the same scale

`critic_score` is out of 100 and `user_score` is out of 10. We make `user_score_100` (user score × 10) so the two can be compared directly. The original `user_score` stays in the file.

In [9]:
df['user_score_100'] = df['user_score'] * 10
df[['critic_score', 'user_score_100']].describe()

,critic_score,user_score_100
count,22350.000000,18890.000000
mean,70.005996,69.598994
std,12.762564,14.183951
min,6.000000,0.000000
25%,63.000000,63.000000
50%,72.000000,73.000000
75%,79.000000,80.000000
max,99.000000,97.000000


## New features

- **`score_gap`** = critic score − user score (both out of 100). Positive means critics liked the game more than users.
- **Regional sales shares** = each region's sales ÷ global sales. Comparing raw sales between PC and consoles is unfair because of the PC undercount, but the *share* of a game's sales in each region is a fairer way to compare.

In [ ]:
df['score_gap'] = df['critic_score'] - df['user_score_100']

for region in ['na', 'eu', 'jp', 'other']:
    df[f'{region}_share'] = df[f'{region}_sales'] / df['global_sales']

df[['score_gap', 'na_share', 'eu_share', 'jp_share', 'other_share']].describe()

## Missing values

We are not filling in missing scores, sales, ESRB ratings, or developers. Making up a rating or a sales number would put fake values into the exact columns we are analyzing.

This table shows how much of each column we actually have:

In [10]:
missing = df.isna().sum().to_frame('missing')
missing['% missing'] = (missing['missing'] / len(df) * 100).round(1)
missing[missing['missing'] > 0].sort_values('missing', ascending=False)

,missing,% missing
steam_genres,26343,86.3
steam_negative,26336,86.3
steam_positive,26336,86.3
steam_critic_score,26336,86.3
steam_year,26336,86.3
steam_appid,26336,86.3
steam_owners_low,26336,86.3
steam_owners_high,26336,86.3
vg_user_count,22941,75.2
vg_user_score,22941,75.2


## Outliers

**Sales:** a few games sold way more than everything else (*Wii Sports* at 82.5 million, *Super Mario Bros.* at 40.2 million). The median game sold only about 0.17 million, so in the analysis we should use medians or a log scale for sales charts so these few games don't hide everything else.

**Scores:** all critic scores are between 0 and 100 and all user scores are between 0 and 10, so there are no impossible values.

In [11]:
display(df.nlargest(10, 'global_sales')[['name', 'platform', 'year', 'global_sales']])
df['global_sales'].describe()

,name,platform,year,global_sales
22528,Wii Sports,Wii,2006.0,82.53
18924,Super Mario Bros.,NES,1985.0,40.24
11137,Mario Kart Wii,Wii,2008.0,35.52
22530,Wii Sports Resort,Wii,2009.0,32.77
14872,Pokemon Red/Pokemon Blue,GB,1996.0,31.37
19616,Tetris,GB,1989.0,30.26
13385,New Super Mario Bros.,DS,2006.0,29.80
22526,Wii Play,Wii,2006.0,28.92
13389,New Super Mario Bros. Wii,Wii,2009.0,28.32
5412,Duck Hunt,NES,1984.0,28.31


count    16710.000000
mean         0.533383
std          1.548143
min          0.010000
25%          0.060000
50%          0.170000
75%          0.470000
max         82.530000
Name: global_sales, dtype: float64

## Final checks and save

In [12]:
print("Rows:", len(df))
print("Duplicate games:", df.duplicated(['name_key', 'platform']).sum())
print("Rows per platform group:")
display(df['platform_group'].value_counts())
print("PC games with a critic score:", ((df['platform_group'] == 'PC') & df['critic_score'].notna()).sum())

Rows: 30526
Duplicate games: 0
Rows per platform group:


platform_group
PlayStation    9741
Nintendo       8599
PC             7281
Xbox           4382
Other           523
Name: count, dtype: int64

PC games with a critic score: 7060


In [13]:
df.to_csv(CLEAN + "games_final.csv", index=False)